# Customer Model Validation

**Why this notebook:** Production pricing depends on **`compute_purchase_probability`** being monotone, bounded, and aligned with the written formula. This notebook is the **visual regression suite** you run before merging changes to [`app/domain/customer.py`](../app/domain/customer.py).

**Audience:** Engineers changing demand logic; reviewers who want assertion-backed evidence.

**Outcome:** Every factor in the purchase model is plotted **and** asserted; Bernoulli sampling is checked against theory.

**Regression test and visual reference for `app/domain/customer.py`.**

This notebook systematically validates every factor in `compute_purchase_probability` and the `decide_purchase` Bernoulli draw. Run it after any change to `Customer`, `PurchaseContext`, or the probability formula to catch regressions before they reach the simulation engine.

| Section | What is validated | Needs DB? |
|---------|-------------------|-----------|
| §0 | Setup & imports | No |
| §1 | Purchase probability formula — component breakdown | No |
| §2 | Budget hard cutoff | No |
| §3 | Price sensitivity curve | No |
| §4 | New vs returning customers | No |
| §5 | Repeat boost accumulation | No |
| §6 | Calendar multipliers (temporal & geographic) | No |
| §7 | Promo eligibility gate | No |
| §8 | Retention score effect | No |
| §9 | Bernoulli draw — empirical vs theoretical | No |

**No database required.** Run from the repo root with the venv active and `pip install -e ".[dev]"`.


## §0  Setup & imports


In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from app.domain.customer import Customer, PurchaseContext

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.35

RNG = np.random.default_rng(42)

# ── Canonical test fixture ─────────────────────────────────────────────────
# A new customer with mid-range parameters. Individual sections will override
# specific fields to isolate each model factor.
BASE_CTX = PurchaseContext(temporal_multiplier=1.0, geographic_multiplier=1.0, promo_eligible=True)

def make_customer(**overrides) -> Customer:
    """Return a fresh Customer from a canonical baseline, applying any overrides."""
    defaults = dict(
        customer_id=1,
        customer_index=0,
        budget=50.0,
        buy_propensity=0.30,
        price_threshold=25.0,
        repeat_boost=0.40,
        basket_mean=18.0,
        location_zone="A",
        retention_sensitivity=0.5,
        promo_sensitivity=0.5,
    )
    defaults.update(overrides)
    return Customer(**defaults)

print("Setup complete — no database required.")


### Follow-up questions

- Why is `make_customer` duplicated or imported here — where is the canonical `Customer` path in production code?
- If assertions pass but results look wrong, what intermediate print would you add first?
- How does this notebook relate to `customer_model_validation` in CI — should it stay fast?



## §1  The purchase probability formula

`compute_purchase_probability` combines five multiplicative factors:

$$p = \text{base} \times \text{price\_effect} \times \text{repeat\_effect} \times \text{calendar} \times \text{retention}$$

| Factor | Formula | Effect |
|--------|---------|--------|
| `base` | `buy_propensity` | Intrinsic purchase likelihood (Beta-distributed across cohort) |
| `price_effect` | `1 / (1 + max(0, ratio − 1))` where `ratio = min(price/threshold, 3)` | Soft price sensitivity — no penalty when price ≤ threshold |
| `repeat_effect` | `1 + repeat_boost × (1 + 0.1 × purchase_count)` if returned; `max(0.15, 1 − 0.45 × promo_sensitivity)` if new | Returning customers get a compounding lift; new customers face a mild haircut |
| `calendar` | `temporal_multiplier × geographic_multiplier × (0.85 if not promo_eligible)` | External demand context |
| `retention` | `1 + 0.15 × (retention_score − 1) × retention_sensitivity` | Recency / loyalty signal |

The final probability is clipped to `[0, 1]`. The **budget** acts as a hard gate: if `offered_price > budget`, the method returns `0.0` immediately, bypassing all factors.


In [ ]:
# Show each factor's contribution for a single canonical scenario
c = make_customer()
ctx = BASE_CTX
price = 20.0

base = c.buy_propensity
ratio = min(price / c.price_threshold, 3.0)
price_effect = 1.0 / (1.0 + max(0.0, ratio - 1.0))
repeat_effect = max(0.15, 1.0 - 0.45 * c.promo_sensitivity)  # new customer
calendar = ctx.temporal_multiplier * ctx.geographic_multiplier  # promo_eligible=True → no penalty
retention = 1.0 + 0.15 * (c.retention_score - 1.0) * c.retention_sensitivity
p_manual = base * price_effect * repeat_effect * calendar * retention
p_api    = c.compute_purchase_probability(price, 1, ctx)

rows = [
    ("base (buy_propensity)",  f"{base:.4f}",          "intrinsic probability"),
    ("price_effect",            f"{price_effect:.4f}",   f"price {price} vs threshold {c.price_threshold}"),
    ("repeat_effect",           f"{repeat_effect:.4f}",  "new customer (no prior purchases)"),
    ("calendar",                f"{calendar:.4f}",       "temporal × geographic × promo"),
    ("retention",               f"{retention:.4f}",      f"retention_score={c.retention_score}, sensitivity={c.retention_sensitivity}"),
    ("→ p (manual product)",    f"{p_manual:.4f}",       "—"),
    ("→ p (API call)",          f"{p_api:.4f}",          "must equal manual"),
]

df = pd.DataFrame(rows, columns=["Factor", "Value", "Notes"])
print(df.to_string(index=False))

assert abs(p_manual - p_api) < 1e-9, "manual computation must match API"
print("\nAssertion passed: manual product matches compute_purchase_probability()")


### Follow-up questions

- Which multiplicative factor in the formula is hardest to explain to a non-modeler?
- Change one factor by 10% — which has the largest impact on the printed probability?
- Where is the same formula implemented for runtime simulation (`app/domain/customer.py`)?



## §2  Budget hard cutoff

If `offered_price > budget`, the customer physically cannot afford the order; `compute_purchase_probability` must return exactly `0.0` regardless of all other factors. Below budget, even a very high price should yield a positive probability.


In [ ]:
c = make_customer(budget=50.0, buy_propensity=0.8, price_threshold=10.0)  # generous propensity

prices = np.linspace(0, 80, 300)
probs  = [c.compute_purchase_probability(float(p), 1, BASE_CTX) for p in prices]

# ── Assertions ────────────────────────────────────────────────────────────
assert c.compute_purchase_probability(50.01, 1, BASE_CTX) == 0.0, "just above budget must be 0"
assert c.compute_purchase_probability(50.00, 1, BASE_CTX) > 0.0, "at budget should be positive"
assert c.compute_purchase_probability(0.01,  1, BASE_CTX) > 0.0, "near-zero price should be positive"
print("Budget cutoff assertions passed")

# ── Plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(prices, probs, lw=2, color="steelblue")
ax.axvline(c.budget, color="crimson", lw=1.5, ls="--", label=f"budget = {c.budget}")
ax.set_xlabel("Offered price ($)")
ax.set_ylabel("Purchase probability")
ax.set_title("§2  Budget hard cutoff — p drops to exactly 0 above budget")
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
plt.tight_layout()
plt.show()


### Follow-up questions

- Why must probability hit zero when basket exceeds budget — is that realistic for your use case?
- Craft a scenario where budget binds despite “high” propensity.
- If you relaxed the budget rule, what downstream metrics would break first?



## §3  Price sensitivity curve

The `price_effect` factor uses a soft sigmoid-like penalty:

$$\text{price\_effect} = \frac{1}{1 + \max(0,\; r - 1)} \qquad r = \min\!\left(\frac{\text{price}}{\text{threshold}},\; 3\right)$$

- When `price ≤ threshold` → `r ≤ 1` → `price_effect = 1.0` (no penalty)
- When `price = 2 × threshold` → `r = 2` → `price_effect = 0.5`
- When `price ≥ 3 × threshold` → `r = 3` → `price_effect = 0.5` (minimum, capped)

The effect is monotonically non-increasing beyond the threshold.


In [ ]:
c = make_customer(budget=200.0, buy_propensity=0.5, price_threshold=30.0)
threshold = c.price_threshold

prices = np.linspace(1, 100, 400)
probs  = [c.compute_purchase_probability(float(p), 1, BASE_CTX) for p in prices]

# Decompose into just the price_effect to visualise it clearly
price_effects = []
for p in prices:
    ratio = min(p / threshold, 3.0)
    price_effects.append(1.0 / (1.0 + max(0.0, ratio - 1.0)))

# ── Assertion: monotonically non-increasing beyond threshold ──────────────
over_threshold = [pe for p, pe in zip(prices, price_effects) if p > threshold]
for i in range(1, len(over_threshold)):
    assert over_threshold[i] <= over_threshold[i - 1] + 1e-9, "price_effect must be non-increasing past threshold"
print("Monotone price_effect assertion passed")

# ── Spot checks ───────────────────────────────────────────────────────────
pe_at_threshold = price_effects[np.argmin(np.abs(prices - threshold))]
pe_at_2x        = price_effects[np.argmin(np.abs(prices - 2 * threshold))]
assert abs(pe_at_threshold - 1.0) < 0.01, "price_effect should be ~1 at threshold"
assert abs(pe_at_2x - 0.5) < 0.02,        "price_effect should be ~0.5 at 2× threshold"
print(f"price_effect at threshold ({threshold}): {pe_at_threshold:.3f} (expected ~1.0)")
print(f"price_effect at 2× threshold ({2*threshold}): {pe_at_2x:.3f} (expected ~0.5)")

# ── Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

ax = axes[0]
ax.plot(prices, price_effects, lw=2, color="darkorange")
ax.axvline(threshold,     color="gray", lw=1.2, ls="--", label=f"threshold = {threshold}")
ax.axvline(2 * threshold, color="gray", lw=1.2, ls=":",  label=f"2× threshold = {2*threshold}")
ax.axhline(1.0, color="steelblue", lw=0.8, ls="-", alpha=0.5, label="price_effect = 1.0")
ax.axhline(0.5, color="steelblue", lw=0.8, ls="-", alpha=0.5, label="price_effect = 0.5")
ax.set_xlabel("Offered price ($)")
ax.set_ylabel("price_effect")
ax.set_title("price_effect factor alone")
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(prices, probs, lw=2, color="steelblue")
ax.axvline(threshold, color="gray", lw=1.2, ls="--", label=f"threshold = {threshold}")
ax.set_xlabel("Offered price ($)")
ax.set_ylabel("Purchase probability")
ax.set_title("Full p — all factors combined")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend(fontsize=8)

fig.suptitle("§3  Price sensitivity curve (threshold = $30, budget = $200)", fontsize=11)
plt.tight_layout()
plt.show()


### Follow-up questions

- From the curve, estimate elasticity informally — where is demand most price-sensitive?
- How does `price_threshold` interact with steepness of the drop?
- Overlay two curves differing only in `buy_propensity` — how do they separate?



## §4  New vs returning customers

A returning customer (`has_purchased_before=True`) receives a **repeat_effect** lift of:

$$\text{repeat\_effect}_{\text{returning}} = 1 + \text{repeat\_boost} \times (1 + 0.1 \times \text{purchase\_count})$$

A new customer gets a dampening:

$$\text{repeat\_effect}_{\text{new}} = \max(0.15,\; 1 - 0.45 \times \text{promo\_sensitivity})$$

At the canonical `promo_sensitivity = 0.5`, the new-customer effect is `0.775`. The difference between new and returning customers is the primary mechanism driving repeat-purchase economics.


In [ ]:
price = 20.0

c_new = make_customer(has_purchased_before=False, purchase_count=0)
c_ret = make_customer(has_purchased_before=True,  purchase_count=3)

p_new = c_new.compute_purchase_probability(price, 1, BASE_CTX)
p_ret = c_ret.compute_purchase_probability(price, 1, BASE_CTX)

# ── Assertions ────────────────────────────────────────────────────────────
assert p_ret > p_new, "returning customer must have higher probability than new customer"
print(f"p_new  = {p_new:.4f}")
print(f"p_ret  = {p_ret:.4f}")
print(f"lift   = {(p_ret - p_new):.4f}  (+{100*(p_ret/p_new - 1):.1f}% relative)")
print("Assertion passed: p_returning > p_new")

# ── Sweep over different prices to show the gap persists ──────────────────
prices = np.linspace(1, c_new.budget, 200)
ps_new = [c_new.compute_purchase_probability(float(p), 1, BASE_CTX) for p in prices]
ps_ret = [c_ret.compute_purchase_probability(float(p), 1, BASE_CTX) for p in prices]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

ax = axes[0]
bars = ax.bar(["New customer", "Returning (3 orders)"], [p_new, p_ret],
              color=["#5BA4CF", "#E07A5F"], width=0.45, edgecolor="white")
for bar, val in zip(bars, [p_new, p_ret]):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.005,
            f"{val:.3f}", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("Purchase probability at $20")
ax.set_title(f"Single price point (${price})")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_ylim(0, max(p_new, p_ret) * 1.3)

ax = axes[1]
ax.plot(prices, ps_new, lw=2, color="#5BA4CF",  label="New customer")
ax.plot(prices, ps_ret, lw=2, color="#E07A5F",  label="Returning (3 orders)")
ax.fill_between(prices, ps_new, ps_ret, alpha=0.15, color="#E07A5F", label="Repeat lift")
ax.axvline(c_new.price_threshold, color="gray", lw=1, ls="--", label="threshold")
ax.set_xlabel("Offered price ($)")
ax.set_ylabel("Purchase probability")
ax.set_title("Across all prices")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend(fontsize=8)

fig.suptitle("§4  New vs returning customer — repeat_effect lift", fontsize=11)
plt.tight_layout()
plt.show()


### Follow-up questions

- Quantify the new vs returning gap at a fixed price — when is it negligible?
- Tie `has_purchased_before` to cohort initialization in the engine.
- What assertion would catch a regression if returning-customer treatment were broken?



## §5  Repeat boost accumulation

For a returning customer, `repeat_effect` grows with `purchase_count`:

$$\text{repeat\_effect} = 1 + \text{repeat\_boost} \times (1 + 0.1 \times \text{purchase\_count})$$

Each additional order increases the multiplier by `0.1 × repeat_boost`. The overall purchase probability should increase monotonically — more orders → higher propensity to buy again.


In [ ]:
price = 20.0
counts = np.arange(0, 25)

def prob_at_count(n: int, boost: float) -> float:
    c = make_customer(has_purchased_before=True, purchase_count=n, repeat_boost=boost)
    return c.compute_purchase_probability(price, 1, BASE_CTX)

boosts = [0.2, 0.4, 0.6]
results = {b: [prob_at_count(n, b) for n in counts] for b in boosts}

# ── Assertions: strictly increasing ──────────────────────────────────────
for b, probs in results.items():
    for i in range(1, len(probs)):
        if probs[i] < 1.0:  # once clipped at 1.0 it can plateau
            assert probs[i] >= probs[i - 1], f"p must be non-decreasing for boost={b}"
print("Monotone accumulation assertions passed for all boost levels")

# ── Plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#5BA4CF", "#E07A5F", "#3D405B"]
for b, color in zip(boosts, colors):
    ax.plot(counts, results[b], lw=2, color=color, marker="o", ms=4, label=f"repeat_boost = {b}")

ax.axhline(1.0, color="gray", lw=1, ls="--", label="p = 1.0 (ceiling)")
ax.set_xlabel("purchase_count (number of prior orders)")
ax.set_ylabel("Purchase probability")
ax.set_title("§5  Repeat boost accumulation — p grows with prior order count")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────
df = pd.DataFrame(
    {f"boost={b}": [f"{v:.3f}" for v in results[b]] for b in boosts},
    index=[f"n={n}" for n in counts]
)
print("\nProbability at each purchase_count:")
df.iloc[::5]  # every 5th row for brevity


### Follow-up questions

- After N purchases, how much lift does `repeat_boost` provide — is it diminishing?
- Where is repeat boost capped or decayed in the full customer lifecycle (see `repeat_purchase_validation.ipynb`)?
- Design an assertion that fails if repeat boost stops accumulating on purchase.



## §6  Calendar multipliers (temporal & geographic)

The `calendar` factor is the product of two external demand signals:

$$\text{calendar} = \text{temporal\_multiplier} \times \text{geographic\_multiplier} \times \begin{cases} 0.85 & \text{if not promo\_eligible} \\ 1.0 & \text{otherwise} \end{cases}$$

These are injected via `PurchaseContext` and are computed by `app/services/pricing/temporal.py` and `app/services/pricing/geographic.py`. Here we test the model's response to the context values directly — confirming proportional scaling.


In [ ]:
c = make_customer(has_purchased_before=False)
price = 20.0

# Sweep temporal_multiplier with geographic fixed
temporal_mults = [0.80, 0.90, 1.00, 1.12, 1.25]
geo_mults      = [0.85, 0.92, 1.00, 1.08, 1.15]

p_temporal = [
    c.compute_purchase_probability(price, 1, PurchaseContext(temporal_multiplier=t, geographic_multiplier=1.0))
    for t in temporal_mults
]
p_geo = [
    c.compute_purchase_probability(price, 1, PurchaseContext(temporal_multiplier=1.0, geographic_multiplier=g))
    for g in geo_mults
]

p_base = c.compute_purchase_probability(price, 1, BASE_CTX)

# ── Assertions: proportional scaling ─────────────────────────────────────
for t, p in zip(temporal_mults, p_temporal):
    expected = min(1.0, p_base * t)
    assert abs(p - expected) < 0.001, f"temporal mult {t}: expected {expected:.4f}, got {p:.4f}"
print("Temporal multiplier proportional scaling: PASSED")

for g, p in zip(geo_mults, p_geo):
    expected = min(1.0, p_base * g)
    assert abs(p - expected) < 0.001, f"geo mult {g}: expected {expected:.4f}, got {p:.4f}"
print("Geographic multiplier proportional scaling: PASSED")

# Confirm promo_eligible=False applies 0.85 penalty
p_promo_off = c.compute_purchase_probability(price, 1, PurchaseContext(promo_eligible=False))
assert abs(p_promo_off - p_base * 0.85) < 0.001, "promo_eligible=False should apply 0.85 factor"
print("Promo-ineligible 0.85 calendar penalty: PASSED")

# ── Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.bar([str(t) for t in temporal_mults], p_temporal, color="#5BA4CF", edgecolor="white")
ax.axhline(p_base, color="crimson", lw=1.5, ls="--", label=f"baseline p = {p_base:.3f}")
ax.set_xlabel("temporal_multiplier")
ax.set_ylabel("Purchase probability")
ax.set_title("Temporal multiplier sweep\n(geo=1.0, promo=True)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend(fontsize=8)

ax = axes[1]
ax.bar([str(g) for g in geo_mults], p_geo, color="#E07A5F", edgecolor="white")
ax.axhline(p_base, color="steelblue", lw=1.5, ls="--", label=f"baseline p = {p_base:.3f}")
ax.set_xlabel("geographic_multiplier")
ax.set_ylabel("Purchase probability")
ax.set_title("Geographic multiplier sweep\n(temporal=1.0, promo=True)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend(fontsize=8)

fig.suptitle("§6  Calendar multipliers — proportional scaling", fontsize=11)
plt.tight_layout()
plt.show()


### Follow-up questions

- When temporal and geographic multipliers combine, which one dominates for your default customer?
- Set multipliers to extremes — can probability exceed 1 before clipping, or is it always bounded?
- Where are these multipliers applied relative to promo gates in the engine order of operations?



## §7  Promo eligibility gate

When `promo_eligible=False`, the `calendar` factor is multiplied by `0.85`. This represents that customers not reachable by promo channels are somewhat less likely to transact on a given day — a deliberate dampener to make the control arm slightly less active on promo days.

The penalty applies irrespective of all other context (temporal, geographic, repeat history, etc.).


In [ ]:
price = 20.0
scenarios = [
    ("New customer",       make_customer(has_purchased_before=False)),
    ("Returning (5 orders)", make_customer(has_purchased_before=True, purchase_count=5)),
    ("High propensity",    make_customer(buy_propensity=0.7)),
    ("Low propensity",     make_customer(buy_propensity=0.1)),
]

ctx_on  = PurchaseContext(promo_eligible=True)
ctx_off = PurchaseContext(promo_eligible=False)

labels, p_on, p_off, ratios = [], [], [], []
for label, c in scenarios:
    p1 = c.compute_purchase_probability(price, 1, ctx_on)
    p0 = c.compute_purchase_probability(price, 1, ctx_off)
    ratio = p0 / p1 if p1 > 0 else 0
    assert abs(ratio - 0.85) < 0.001 or p1 == 0.0, f"{label}: expected 0.85 ratio, got {ratio:.4f}"
    labels.append(label)
    p_on.append(p1)
    p_off.append(p0)
    ratios.append(ratio)

print("Promo gate 0.85 penalty assertions passed for all scenarios")
print()
df = pd.DataFrame({"Scenario": labels, "promo_eligible=True": [f"{v:.4f}" for v in p_on],
                   "promo_eligible=False": [f"{v:.4f}" for v in p_off],
                   "Ratio (off/on)": [f"{r:.3f}" for r in ratios]})
print(df.to_string(index=False))

# ── Plot ──────────────────────────────────────────────────────────────────
x = np.arange(len(labels))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
b1 = ax.bar(x - w/2, p_on,  w, label="promo_eligible=True",  color="#5BA4CF", edgecolor="white")
b2 = ax.bar(x + w/2, p_off, w, label="promo_eligible=False", color="#E07A5F", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("Purchase probability")
ax.set_title("§7  Promo eligibility gate — 15% demand dampener when not promo-eligible")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


### Follow-up questions

- Walk through a customer who is almost eligible but fails one gate — which gate is it?
- How would promo eligibility interact with experiment arms that change fees but not promos?
- Add an assertion for campaign budget exhaustion — what state variables would you need?



## §8  Retention score effect

The `retention` factor modulates purchase probability based on how recently a customer has ordered:

$$\text{retention} = 1 + 0.15 \times (\text{retention\_score} - 1) \times \text{retention\_sensitivity}$$

- **Floor:** when `retention_score = 1.0` (fresh/decayed customer), `retention = 1.0` — no effect
- **Ceiling:** when `retention_score = 2.5` (maximum accumulation), `retention = 1 + 0.15 × 1.5 × sensitivity`
- At `retention_sensitivity = 0.5`, the maximum `retention` multiplier is `1.1125` — a modest 11% lift

The score is set by `register_purchase` and decayed by `decay_retention` (validated in detail in `repeat_purchase_validation.ipynb`).


In [ ]:
price = 20.0
retention_scores = np.linspace(1.0, 2.5, 100)
sensitivities = [0.25, 0.50, 1.00]

results = {}
for sens in sensitivities:
    probs = []
    for rs in retention_scores:
        c = make_customer(retention_sensitivity=sens, retention_score=rs,
                          has_purchased_before=True, purchase_count=1)
        probs.append(c.compute_purchase_probability(price, 1, BASE_CTX))
    results[sens] = probs

# ── Assertions ────────────────────────────────────────────────────────────
for sens, probs in results.items():
    for i in range(1, len(probs)):
        assert probs[i] >= probs[i - 1] - 1e-9, f"p must be non-decreasing with retention_score (sens={sens})"
print("Monotone retention score assertions passed for all sensitivity levels")

# Boundary check: at score=1.0, retention multiplier = 1.0 for all sensitivities
for sens in sensitivities:
    c_floor = make_customer(retention_sensitivity=sens, retention_score=1.0,
                            has_purchased_before=True, purchase_count=1)
    c_ceil  = make_customer(retention_sensitivity=sens, retention_score=2.5,
                            has_purchased_before=True, purchase_count=1)
    p_floor = c_floor.compute_purchase_probability(price, 1, BASE_CTX)
    p_ceil  = c_ceil.compute_purchase_probability(price, 1, BASE_CTX)
    expected_mult = 1.0 + 0.15 * (2.5 - 1.0) * sens
    assert abs(p_ceil / p_floor - expected_mult) < 0.001, f"max lift mismatch for sens={sens}"
    print(f"retention_sensitivity={sens}: p(score=1.0)={p_floor:.4f}, p(score=2.5)={p_ceil:.4f}, "
          f"max lift={expected_mult:.4f}")

# ── Plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#5BA4CF", "#E07A5F", "#3D405B"]
for sens, color in zip(sensitivities, colors):
    ax.plot(retention_scores, results[sens], lw=2, color=color, label=f"retention_sensitivity = {sens}")

ax.axvline(1.0, color="gray", lw=1, ls="--", alpha=0.5, label="score floor = 1.0")
ax.axvline(2.5, color="gray", lw=1, ls=":",  alpha=0.5, label="score ceiling = 2.5")
ax.set_xlabel("retention_score")
ax.set_ylabel("Purchase probability")
ax.set_title("§8  Retention score effect on purchase probability")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


### Follow-up questions

- How does retention score shift purchase probability relative to price effects — which wins in your plots?
- At churn floor vs ceiling retention, what is churn hazard (see `docs/mathematical-models.md`)?
- Connect retention dynamics to CLV predictions in `app/domain/customer.py`.



## §9  Bernoulli draw — empirical vs theoretical

`decide_purchase` wraps `compute_purchase_probability` in a single Bernoulli trial:

```python
return random_state.random() < p
```

By the Law of Large Numbers, the empirical purchase rate over `N` independent draws should converge to the theoretical `p`. We validate this across a diverse set of scenarios with `N = 5 000` draws each. The acceptance criterion is `|empirical − theoretical| < 0.03` (within 3 percentage points).


In [ ]:
N = 5_000
rng = np.random.default_rng(99)

scenarios = [
    ("Low propensity, new",       make_customer(buy_propensity=0.1, has_purchased_before=False), 20.0, BASE_CTX),
    ("Mid propensity, new",       make_customer(buy_propensity=0.3, has_purchased_before=False), 20.0, BASE_CTX),
    ("High propensity, new",      make_customer(buy_propensity=0.6, has_purchased_before=False), 20.0, BASE_CTX),
    ("Returning, 5 orders",       make_customer(buy_propensity=0.3, has_purchased_before=True,  purchase_count=5), 20.0, BASE_CTX),
    ("Weekend demand boost",      make_customer(buy_propensity=0.3),                             20.0, PurchaseContext(temporal_multiplier=1.12)),
    ("Promo-ineligible",          make_customer(buy_propensity=0.3),                             20.0, PurchaseContext(promo_eligible=False)),
    ("At price threshold",        make_customer(buy_propensity=0.4, price_threshold=20.0),       20.0, BASE_CTX),
    ("Price 2× threshold",        make_customer(buy_propensity=0.4, price_threshold=10.0, budget=100.0), 20.0, BASE_CTX),
]

rows = []
for label, c, price, ctx in scenarios:
    p_theory = c.compute_purchase_probability(price, 1, ctx)
    draws = sum(c.decide_purchase(rng, price, 1, ctx) for _ in range(N))
    p_empirical = draws / N
    error = abs(p_empirical - p_theory)
    passed = error < 0.03
    assert passed, f"{label}: |{p_empirical:.4f} - {p_theory:.4f}| = {error:.4f} ≥ 0.03"
    rows.append((label, p_theory, p_empirical, error, "PASS" if passed else "FAIL"))

df = pd.DataFrame(rows, columns=["Scenario", "Theoretical p", "Empirical p (N=5000)", "|Error|", "Status"])
df["Theoretical p"] = df["Theoretical p"].map("{:.4f}".format)
df["Empirical p (N=5000)"] = df["Empirical p (N=5000)"].map("{:.4f}".format)
df["|Error|"] = df["|Error|"].map("{:.4f}".format)
print(df.to_string(index=False))
print("\nAll Bernoulli draw convergence assertions passed")

# ── Plot ──────────────────────────────────────────────────────────────────
labels_short = [r[0] for r in rows]
theory_vals  = [c.compute_purchase_probability(p, 1, ctx) for _, c, p, ctx in scenarios]
empirical_vals = [float(r[2]) for r in rows]
errors_vals    = [float(r[3]) for r in rows]

x = np.arange(len(labels_short))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(x - w/2, theory_vals,   w, label="Theoretical p",      color="#5BA4CF",  edgecolor="white")
ax.bar(x + w/2, empirical_vals, w, label="Empirical (N=5 000)", color="#E07A5F", edgecolor="white",
       yerr=errors_vals, capsize=4, error_kw=dict(elinewidth=1.5, ecolor="#555"))
ax.set_xticks(x)
ax.set_xticklabels([l.replace(", ", "\n") for l in labels_short], fontsize=8)
ax.set_ylabel("Purchase probability")
ax.set_title("§9  Bernoulli draw — empirical rate converges to theoretical p (N=5 000)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


### Follow-up questions

- Is empirical frequency within sampling noise of p — run larger N and describe the CLT behavior.
- What seed would you fix in CI to keep this cell deterministic?
- If means differ systematically, is the bug more likely in `rng.binomial` usage or in `compute_purchase_probability`?



## Summary

All assertions in this notebook are inline and will raise `AssertionError` on regression. The table below summarises what each section tests:

| Section | Assertion tested |
|---------|------------------|
| §1 | Manual factor product == `compute_purchase_probability()` |
| §2 | `p == 0.0` above budget; `p > 0` at or below budget |
| §3 | `price_effect` monotonically non-increasing past threshold; `~1.0` at threshold, `~0.5` at 2× threshold |
| §4 | `p_returning > p_new` at same price |
| §5 | `p` non-decreasing as `purchase_count` grows (all boost levels) |
| §6 | Temporal and geographic multipliers scale `p` proportionally; promo-off applies 0.85 |
| §7 | `promo_eligible=False` reduces `p` by exactly 15% across all customer types |
| §8 | `p` non-decreasing with `retention_score`; max lift matches formula |
| §9 | `|empirical − theoretical| < 0.03` for 8 scenarios at N=5 000 |


### Pytest mirror & next up

- Mirror critical assertions in `tests/` for CI — from the repo root: `pytest tests/ -q`.
- **Next:** [`repeat_purchase_validation.ipynb`](repeat_purchase_validation.ipynb) — multi-day retention dynamics that feed the same purchase model.



In [ ]:
# Automated mirror — run in a terminal from the repo root (venv active):
#   pytest tests/ -q
print("pytest tests/ -q")
